# 09. QLoRA Fine-tuning for Civil Complaint Domain

QLoRA 4-bit 양자화로 Qwen2.5-7B-Instruct를 민원 도메인에 특화 학습합니다.
Base model vs Fine-tuned model의 답변 품질을 비교 평가합니다.

| Item | Detail |
|------|--------|
| Task | 민원 도메인 QLoRA Fine-tuning |
| Base Model | Qwen/Qwen2.5-7B-Instruct |
| Method | QLoRA (4-bit NF4 + LoRA r=16) |
| Data | train_llm.parquet / val_llm.parquet |
| Metrics | ROUGE-L, BERTScore |
| Environment | Kaggle T4 x2 GPU |

---
## 1. Environment Setup

In [ ]:
%%capture
!pip install -q transformers>=4.40 accelerate>=0.28 bitsandbytes>=0.43 peft>=0.10 trl>=0.8 datasets plotly rouge-score bert-score sentencepiece kaleido

In [ ]:
import os, json, time, warnings, gc
import numpy as np
import pandas as pd
import torch
from transformers import (
    AutoModelForCausalLM, AutoTokenizer,
    BitsAndBytesConfig, TrainingArguments,
    EarlyStoppingCallback,
)
from peft import LoraConfig, get_peft_model, PeftModel, prepare_model_for_kbit_training
from trl import SFTTrainer
from datasets import Dataset
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import plotly.io as pio
pio.renderers.default = 'iframe'
from IPython.display import display, HTML
warnings.filterwarnings('ignore')

# --- GPU 최적화 ---
if torch.cuda.is_available():
    torch.backends.cudnn.benchmark = True
    torch.set_float32_matmul_precision('medium')
    for i in range(torch.cuda.device_count()):
        name = torch.cuda.get_device_name(i)
        mem = torch.cuda.get_device_properties(i).total_mem / 1e9
        print(f"  GPU {i}: {name} ({mem:.1f} GB)")
    print(f"  cuDNN benchmark = True, float32 matmul precision = medium")

if os.path.exists('/kaggle/input'):
    DATA_DIR = '/kaggle/input/civilcomplaint-processed'
    OUT_DIR = '/kaggle/working'
    IS_KAGGLE = True
else:
    DATA_DIR = '../data/processed'
    OUT_DIR = '..'
    IS_KAGGLE = False

RESULTS_DIR = os.path.join(OUT_DIR, 'results')
MODELS_DIR = os.path.join(OUT_DIR, 'models')
ADAPTER_DIR = os.path.join(MODELS_DIR, 'lora_adapter')
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(ADAPTER_DIR, exist_ok=True)

print(f"Environment: {'Kaggle' if IS_KAGGLE else 'Local'}")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

# Load RAG results
rag_results = None
rag_path = os.path.join(RESULTS_DIR, 'generation_rag_results.json')
if os.path.exists(rag_path):
    with open(rag_path) as f:
        rag_results = json.load(f)
    print(f"\nRAG results loaded: ROUGE-L={rag_results['metrics'].get('rag_rouge_l', 'N/A')}")

---
## 2. Data Loading

In [ ]:
# --- Load LLM training data ---
train_path = os.path.join(DATA_DIR, 'train_llm.parquet')
val_path = os.path.join(DATA_DIR, 'val_llm.parquet')

train_df = pd.read_parquet(train_path)
val_df = pd.read_parquet(val_path)

print(f"Train: {len(train_df):,} samples")
print(f"Val:   {len(val_df):,} samples")
print(f"Columns: {list(train_df.columns)}")
train_df.head(3)

In [ ]:
# --- Convert to Alpaca-style instruction format ---

def format_alpaca(row):
    """Format QA pair into Alpaca-style instruction text."""
    question = row.get('question', row.get('input', ''))
    answer = row.get('answer', row.get('output', ''))
    domain = row.get('domain', '')
    
    text = f"""### 질문
{question}

### 답변
{answer}"""
    return text


train_df['text'] = train_df.apply(format_alpaca, axis=1)
val_df['text'] = val_df.apply(format_alpaca, axis=1)

# Create HF Datasets
train_dataset = Dataset.from_pandas(train_df[['text']].reset_index(drop=True))
val_dataset = Dataset.from_pandas(val_df[['text']].reset_index(drop=True))

print(f"Train dataset: {len(train_dataset):,}")
print(f"Val dataset:   {len(val_dataset):,}")
print(f"\n--- Sample ---")
print(train_dataset[0]['text'][:500])

In [ ]:
# --- Text length distribution ---
train_lengths = train_df['text'].str.len()
val_lengths = val_df['text'].str.len()

fig = make_subplots(rows=1, cols=2, subplot_titles=['Train', 'Validation'])

fig.add_trace(go.Histogram(
    x=train_lengths, nbinsx=50, name='Train',
    marker_color='#42a5f5', opacity=0.8,
), row=1, col=1)

fig.add_trace(go.Histogram(
    x=val_lengths, nbinsx=50, name='Val',
    marker_color='#66bb6a', opacity=0.8,
), row=1, col=2)

fig.update_layout(
    title='Text Length Distribution (characters)',
    width=900, height=400,
    showlegend=False,
)
fig.update_xaxes(title_text='Characters', row=1, col=1)
fig.update_xaxes(title_text='Characters', row=1, col=2)
fig.show(renderer='iframe')

print(f"Train: mean={train_lengths.mean():.0f}, median={train_lengths.median():.0f}, max={train_lengths.max()}")
print(f"Val:   mean={val_lengths.mean():.0f}, median={val_lengths.median():.0f}, max={val_lengths.max()}")

---
## 3. Model Loading (4-bit Quantization)

In [ ]:
# --- BitsAndBytes 4-bit config ---
MODEL_ID = "Qwen/Qwen2.5-7B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

print(f"Loading {MODEL_ID} with 4-bit quantization...")
t0 = time.time()

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
    torch_dtype=torch.bfloat16,
)

load_time = time.time() - t0
print(f"Model loaded in {load_time:.1f}s")
print(f"Model dtype: {model.dtype}")
print(f"Tokenizer vocab size: {len(tokenizer):,}")

# Model memory
if torch.cuda.is_available():
    mem_gb = torch.cuda.memory_allocated() / 1e9
    print(f"GPU memory used: {mem_gb:.1f} GB")

---
## 4. LoRA Configuration

In [ ]:
# --- LoRA configuration ---
model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)

# Print trainable parameters
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in model.parameters())
trainable_pct = 100 * trainable_params / total_params

print(f"LoRA Config:")
print(f"  r={lora_config.r}, alpha={lora_config.lora_alpha}")
print(f"  Target modules: {lora_config.target_modules}")
print(f"  Dropout: {lora_config.lora_dropout}")
print(f"\nParameters:")
print(f"  Total:     {total_params:,}")
print(f"  Trainable: {trainable_params:,} ({trainable_pct:.2f}%)")

---
## 5. Base Model Evaluation (Before Training)

In [ ]:
# --- Base model evaluation (before training) ---
from rouge_score import rouge_scorer
from bert_score import score as bert_score_fn

N_EVAL = min(200, len(val_df))
eval_subset = val_df.head(N_EVAL).copy()

def generate_batch(model, tokenizer, prompts, max_new_tokens=256):
    """Generate answers for a list of prompts."""
    answers = []
    model.eval()
    with torch.no_grad():
        for prompt in prompts:
            inputs = tokenizer(prompt, return_tensors="pt", truncation=True,
                              max_length=512).to(model.device)
            outputs = model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=False,
                temperature=1.0,
                pad_token_id=tokenizer.eos_token_id,
            )
            generated = tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:],
                                          skip_special_tokens=True)
            answers.append(generated.strip())
    return answers

# Build prompts
base_prompts = []
for _, row in eval_subset.iterrows():
    q = row.get('question', row.get('input', ''))
    base_prompts.append(f"### 질문\n{q}\n\n### 답변\n")

print(f"Evaluating base model on {N_EVAL} samples...")
t0 = time.time()
base_answers = generate_batch(model, tokenizer, base_prompts, max_new_tokens=256)
base_eval_time = time.time() - t0
print(f"Base evaluation done: {base_eval_time:.1f}s ({base_eval_time/N_EVAL:.1f}s/sample)")

# Compute metrics
gt_answers = eval_subset.get('answer', eval_subset.get('output', pd.Series([''] * N_EVAL))).tolist()
scorer = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=False)

base_rouge = [scorer.score(gt, pred)['rougeL'].fmeasure 
              for gt, pred in zip(gt_answers, base_answers)]

try:
    _, _, F1_base = bert_score_fn(base_answers, gt_answers, lang='ko', verbose=False, batch_size=16)
    base_bertscore = F1_base.numpy().tolist()
except Exception:
    base_bertscore = [0.0] * N_EVAL

print(f"\nBase Model Results:")
print(f"  ROUGE-L:     {np.mean(base_rouge):.4f} (\u00b1{np.std(base_rouge):.4f})")
print(f"  BERTScore:   {np.mean(base_bertscore):.4f} (\u00b1{np.std(base_bertscore):.4f})")

---
## 6. Training

In [ ]:
# --- Training arguments ---
PATIENCE = 2  # Early stopping patience

training_args = TrainingArguments(
    output_dir=os.path.join(OUT_DIR, 'checkpoints'),
    num_train_epochs=5,  # 최대 5 epoch (early stopping으로 조기 종료)
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,
    weight_decay=0.01,
    bf16=True,
    logging_steps=10,
    save_strategy="epoch",
    eval_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    report_to="none",
    optim="paged_adamw_8bit",
    max_grad_norm=0.3,
    group_by_length=True,
    remove_unused_columns=False,
)

# --- SFTTrainer with Early Stopping ---
trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    processing_class=tokenizer,
    max_seq_length=1024,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=PATIENCE)],
)

print("Starting training (early stopping patience={PATIENCE})...")
t0 = time.time()
train_result = trainer.train()
train_time = time.time() - t0

print(f"\nTraining complete: {train_time:.0f}s")
print(f"  Train loss: {train_result.training_loss:.4f}")
print(f"  Train samples: {train_result.metrics.get('train_samples_per_second', 0):.1f}/s")
print(f"  Stopped at epoch: {trainer.state.epoch:.0f}/{int(training_args.num_train_epochs)}")

In [ ]:
# --- Training loss curve ---
log_history = trainer.state.log_history

train_steps = [h['step'] for h in log_history if 'loss' in h]
train_losses = [h['loss'] for h in log_history if 'loss' in h]
eval_steps = [h['step'] for h in log_history if 'eval_loss' in h]
eval_losses = [h['eval_loss'] for h in log_history if 'eval_loss' in h]

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=train_steps, y=train_losses,
    mode='lines', name='Train Loss',
    line=dict(color='#42a5f5', width=2),
))
if eval_losses:
    fig.add_trace(go.Scatter(
        x=eval_steps, y=eval_losses,
        mode='lines+markers', name='Eval Loss',
        line=dict(color='#ef5350', width=2),
        marker=dict(size=8),
    ))

fig.update_layout(
    title='Training Loss Curve',
    xaxis_title='Step', yaxis_title='Loss',
    width=800, height=450,
    legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='right', x=1),
)
fig.show(renderer='iframe')

---
## 7. Fine-tuned Model Evaluation

In [ ]:
# --- Fine-tuned model evaluation ---
print(f"Evaluating fine-tuned model on {N_EVAL} samples...")
t0 = time.time()
ft_answers = generate_batch(model, tokenizer, base_prompts, max_new_tokens=256)
ft_eval_time = time.time() - t0
print(f"Fine-tuned evaluation done: {ft_eval_time:.1f}s")

# Compute metrics
ft_rouge = [scorer.score(gt, pred)['rougeL'].fmeasure
            for gt, pred in zip(gt_answers, ft_answers)]

try:
    _, _, F1_ft = bert_score_fn(ft_answers, gt_answers, lang='ko', verbose=False, batch_size=16)
    ft_bertscore = F1_ft.numpy().tolist()
except Exception:
    ft_bertscore = [0.0] * N_EVAL

print(f"\nFine-tuned Model Results:")
print(f"  ROUGE-L:     {np.mean(ft_rouge):.4f} (\u00b1{np.std(ft_rouge):.4f})")
print(f"  BERTScore:   {np.mean(ft_bertscore):.4f} (\u00b1{np.std(ft_bertscore):.4f})")

---
## 8. Comparison

In [ ]:
# --- Base vs Fine-tuned comparison ---
metrics = {
    'ROUGE-L': [np.mean(base_rouge), np.mean(ft_rouge)],
    'BERTScore F1': [np.mean(base_bertscore), np.mean(ft_bertscore)],
}

fig = go.Figure()
fig.add_trace(go.Bar(
    x=list(metrics.keys()),
    y=[v[0] for v in metrics.values()],
    name='Base Model',
    marker_color='#ef5350',
    text=[f'{v[0]:.4f}' for v in metrics.values()],
    textposition='outside',
))
fig.add_trace(go.Bar(
    x=list(metrics.keys()),
    y=[v[1] for v in metrics.values()],
    name='Fine-tuned (LoRA)',
    marker_color='#42a5f5',
    text=[f'{v[1]:.4f}' for v in metrics.values()],
    textposition='outside',
))

fig.update_layout(
    title='Base Model vs Fine-tuned (QLoRA): Generation Quality',
    yaxis_title='Score', yaxis_range=[0, 1.1],
    barmode='group',
    width=700, height=450,
    legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='right', x=1),
)
fig.show(renderer='iframe')

In [ ]:
# --- Score distribution violin ---
fig = make_subplots(rows=1, cols=2, subplot_titles=['ROUGE-L', 'BERTScore F1'])

fig.add_trace(go.Violin(y=base_rouge, name='Base', line_color='#ef5350',
                         box_visible=True, meanline_visible=True, side='negative'), row=1, col=1)
fig.add_trace(go.Violin(y=ft_rouge, name='LoRA', line_color='#42a5f5',
                         box_visible=True, meanline_visible=True, side='positive'), row=1, col=1)

fig.add_trace(go.Violin(y=base_bertscore, name='Base', line_color='#ef5350',
                         box_visible=True, meanline_visible=True, side='negative',
                         showlegend=False), row=1, col=2)
fig.add_trace(go.Violin(y=ft_bertscore, name='LoRA', line_color='#42a5f5',
                         box_visible=True, meanline_visible=True, side='positive',
                         showlegend=False), row=1, col=2)

fig.update_layout(
    title='Score Distribution: Base vs Fine-tuned',
    width=900, height=450,
    legend=dict(orientation='h', yanchor='bottom', y=1.05, xanchor='right', x=1),
)
fig.show(renderer='iframe')

In [ ]:
# --- Qualitative comparison: 5 samples ---
html_parts = ['<h3>Qualitative Comparison: Base vs Fine-tuned (LoRA)</h3>']

for i in range(min(5, N_EVAL)):
    gt = gt_answers[i]
    base_ans = base_answers[i]
    ft_ans = ft_answers[i]
    q = eval_subset.iloc[i].get('question', eval_subset.iloc[i].get('input', ''))
    
    html_parts.append(f'''
    <div style="margin:20px 0; padding:15px; border:1px solid #ddd; border-radius:8px;">
        <b>Query {i+1}:</b> {q}<br>
        <b>Ground Truth:</b> <span style="color:#2e7d32;">{gt[:300]}{"..." if len(gt)>300 else ""}</span><br><br>
        <table style="width:100%; border-collapse:collapse;">
            <tr>
                <td style="width:50%; padding:10px; border:1px solid #eee; vertical-align:top;">
                    <b style="color:#ef5350;">Base Model</b> (ROUGE-L: {base_rouge[i]:.3f})<br>
                    {base_ans[:300]}{"..." if len(str(base_ans))>300 else ""}
                </td>
                <td style="width:50%; padding:10px; border:1px solid #eee; vertical-align:top;">
                    <b style="color:#42a5f5;">Fine-tuned (LoRA)</b> (ROUGE-L: {ft_rouge[i]:.3f})<br>
                    {ft_ans[:300]}{"..." if len(str(ft_ans))>300 else ""}
                </td>
            </tr>
        </table>
    </div>
    ''')

display(HTML(''.join(html_parts)))

---
## 9. Save Adapter & Results

In [ ]:
# --- Save LoRA adapter ---
print(f"Saving LoRA adapter to {ADAPTER_DIR}...")
model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)

# Check saved files
import glob as _glob
saved_files = _glob.glob(os.path.join(ADAPTER_DIR, '*'))
print(f"\nSaved files:")
for f in saved_files:
    size_mb = os.path.getsize(f) / 1024 / 1024
    print(f"  {os.path.basename(f)} ({size_mb:.1f} MB)")

In [ ]:
# --- Save generation_lora_results.json ---
lora_output = {
    'method': 'QLoRA',
    'base_model': MODEL_ID,
    'lora_r': lora_config.r,
    'lora_alpha': lora_config.lora_alpha,
    'target_modules': list(lora_config.target_modules),
    'trainable_params': trainable_params,
    'trainable_pct': round(trainable_pct, 2),
    'training': {
        'epochs': int(training_args.num_train_epochs),
        'batch_size': training_args.per_device_train_batch_size,
        'grad_accum': training_args.gradient_accumulation_steps,
        'lr': training_args.learning_rate,
        'train_loss': round(train_result.training_loss, 4),
        'train_time_s': round(train_time, 0),
    },
    'n_eval': N_EVAL,
    'metrics': {
        'base_rouge_l': round(float(np.mean(base_rouge)), 4),
        'lora_rouge_l': round(float(np.mean(ft_rouge)), 4),
        'base_bertscore': round(float(np.mean(base_bertscore)), 4),
        'lora_bertscore': round(float(np.mean(ft_bertscore)), 4),
    },
    'improvement': {
        'rouge_l_delta': round(float(np.mean(ft_rouge) - np.mean(base_rouge)), 4),
        'bertscore_delta': round(float(np.mean(ft_bertscore) - np.mean(base_bertscore)), 4),
    },
    # Pipeline context
    'rag_baseline': {
        'rag_rouge_l': rag_results['metrics'].get('rag_rouge_l') if rag_results else None,
        'rag_bertscore': rag_results['metrics'].get('rag_bertscore') if rag_results else None,
    },
}

results_path = os.path.join(RESULTS_DIR, 'generation_lora_results.json')
with open(results_path, 'w', encoding='utf-8') as f:
    json.dump(lora_output, f, ensure_ascii=False, indent=2)

print(f"Results saved: {results_path}")
print(json.dumps(lora_output, ensure_ascii=False, indent=2))

In [ ]:
# --- Base64 download ---
import base64, zipfile, io

def create_download_link(filepath, filename=None):
    if filename is None:
        filename = filepath.split('/')[-1]
    with open(filepath, 'rb') as f:
        data = f.read()
    b64 = base64.b64encode(data).decode()
    size_mb = len(data) / 1024 / 1024
    href = (f'<a href="data:application/octet-stream;base64,{b64}" '
            f'download="{filename}">'
            f'Download: {filename} ({size_mb:.1f} MB)</a>')
    display(HTML(href))


def create_zip_download(file_dict, zip_name="artifacts.zip"):
    buffer = io.BytesIO()
    with zipfile.ZipFile(buffer, 'w', zipfile.ZIP_DEFLATED) as zf:
        for arcname, filepath in file_dict.items():
            if os.path.isfile(filepath):
                zf.write(filepath, arcname)
    buffer.seek(0)
    data = buffer.read()
    b64 = base64.b64encode(data).decode()
    size_mb = len(data) / 1024 / 1024
    href = (f'<a href="data:application/octet-stream;base64,{b64}" '
            f'download="{zip_name}">'
            f'Download: {zip_name} ({size_mb:.1f} MB)</a>')
    display(HTML(href))


# Results JSON
create_download_link(results_path)

# LoRA adapter ZIP
adapter_files = {}
for fp in _glob.glob(os.path.join(ADAPTER_DIR, '**', '*'), recursive=True):
    if os.path.isfile(fp):
        rel = os.path.relpath(fp, ADAPTER_DIR)
        adapter_files[f'lora_adapter/{rel}'] = fp

if adapter_files:
    create_zip_download(adapter_files, zip_name="lora_adapter.zip")

---
## 10. Summary

In [ ]:
# --- Progressive improvement chart ---
stages = []
scores = []
colors = []

# RAG baseline
if rag_results:
    stages.append('RAG\n(Ollama)')
    scores.append(rag_results['metrics'].get('rag_bertscore', 0))
    colors.append('#ef5350')

stages.append('Base Model\n(Qwen2.5-7B)')
scores.append(float(np.mean(base_bertscore)))
colors.append('#ffa726')

stages.append('QLoRA\n(Fine-tuned)')
scores.append(float(np.mean(ft_bertscore)))
colors.append('#66bb6a')

fig = go.Figure()
fig.add_trace(go.Bar(
    x=stages, y=scores,
    marker_color=colors,
    text=[f'{v:.4f}' for v in scores],
    textposition='outside',
))

fig.update_layout(
    title='Pipeline Progression: Generation Quality (BERTScore F1)',
    yaxis_title='BERTScore F1',
    yaxis_range=[0, max(scores) * 1.3 if scores else 1],
    width=700, height=450,
)
fig.show(renderer='iframe')

# Summary
print("=" * 60)
print("     09. QLoRA Fine-tuning -- Summary")
print("=" * 60)
print()
print(f"  Base Model:       {MODEL_ID}")
print(f"  LoRA:             r={lora_config.r}, alpha={lora_config.lora_alpha}")
print(f"  Trainable params: {trainable_params:,} ({trainable_pct:.2f}%)")
print(f"  Training loss:    {train_result.training_loss:.4f}")
print(f"  Training time:    {train_time:.0f}s")
print()
print(f"  {'Metric':<20} {'Base':>10} {'LoRA':>10} {'Delta':>10}")
print(f"  {'-'*50}")
print(f"  {'ROUGE-L':<20} {np.mean(base_rouge):>10.4f} {np.mean(ft_rouge):>10.4f} {np.mean(ft_rouge)-np.mean(base_rouge):>+10.4f}")
print(f"  {'BERTScore F1':<20} {np.mean(base_bertscore):>10.4f} {np.mean(ft_bertscore):>10.4f} {np.mean(ft_bertscore)-np.mean(base_bertscore):>+10.4f}")
print()
print(f"  Adapter: {ADAPTER_DIR}")
print(f"  Results: {results_path}")
print()
print("  Next -> 10_gguf_deploy")
print("=" * 60)

---
## 11. QLoRA 하이퍼파라미터 선택 근거

### 양자화 설정

| Parameter | Value | Rationale |
|-----------|-------|-----------|
| `load_in_4bit` | True | 7B 모델의 GPU 메모리를 ~4GB로 축소 (FP16 대비 ~75% 절감). T4 16GB에서 학습 가능 |
| `bnb_4bit_quant_type` | nf4 | NormalFloat4: 가중치 분포가 정규분포에 가까운 Transformer에 최적화된 양자화 방식 (QLoRA 논문 제안) |
| `bnb_4bit_compute_dtype` | bfloat16 | 연산 정밀도는 bf16 유지 → 양자화로 인한 성능 손실 최소화 |
| `bnb_4bit_use_double_quant` | True | 양자화 상수도 양자화 → 추가 메모리 절감 (~0.4GB/7B) |

### LoRA 설정

| Parameter | Value | Rationale |
|-----------|-------|-----------|
| `r` | 16 | Rank 16: 7B 모델에서 충분한 표현력 확보. r=8은 과소, r=32는 과대 파라미터 (경험적 최적) |
| `lora_alpha` | 32 | alpha/r = 2.0 (스케일링 팩터). alpha=2r은 LoRA 논문의 표준 설정 |
| `target_modules` | q,k,v,o,gate,up,down | 모든 attention + FFN 레이어 적용. Attention만 적용 시 대비 FFN 포함이 민원 도메인 적응에 유리 |
| `lora_dropout` | 0.05 | 낮은 dropout: LoRA 자체가 low-rank 정규화 역할 → 추가 dropout은 최소한으로 |

### 학습 설정

| Parameter | Value | Rationale |
|-----------|-------|-----------|
| `lr` | 2e-4 | LoRA fine-tuning 표준 학습률. Full fine-tuning(2e-5) 대비 10배 높음 — LoRA 파라미터만 업데이트하므로 |
| `epochs` | 2 | LLM fine-tuning은 1~3 epoch이 최적. 과적합 방지 + 도메인 적응의 균형 |
| `batch × grad_accum` | 4 × 4 = 16 | 실효 배치 크기 16. GPU 메모리 제약 내 최대 배치 |
| `lr_scheduler` | cosine | Cosine annealing: 학습 후반부 학습률을 부드럽게 감소 → 최적점 수렴 안정화 |
| `optim` | paged_adamw_8bit | 8bit optimizer로 메모리 절감. paged는 GPU OOM 시 CPU offload 지원 |

In [ ]:
# === Base → LoRA 개선율 + RAG Baseline 대비 분석 ===
print("=" * 70)
print("  Generation Quality 개선율 분석")
print("=" * 70)

# Base → LoRA
base_rl = float(np.mean(base_rouge))
lora_rl = float(np.mean(ft_rouge))
base_bs = float(np.mean(base_bertscore))
lora_bs = float(np.mean(ft_bertscore))

print(f"\n  [1] Base Model → QLoRA Fine-tuned")
print(f"  {'Metric':<18} {'Base':>10} {'LoRA':>10} {'Delta':>10} {'Relative':>10}")
print(f"  {'-'*60}")
rl_delta = lora_rl - base_rl
rl_rel = (rl_delta / base_rl * 100) if base_rl > 0 else 0
bs_delta = lora_bs - base_bs
bs_rel = (bs_delta / base_bs * 100) if base_bs > 0 else 0
print(f"  {'ROUGE-L':<18} {base_rl:>10.4f} {lora_rl:>10.4f} {rl_delta:>+10.4f} {rl_rel:>+9.1f}%")
print(f"  {'BERTScore F1':<18} {base_bs:>10.4f} {lora_bs:>10.4f} {bs_delta:>+10.4f} {bs_rel:>+9.1f}%")

# RAG baseline comparison
if rag_results:
    rag_rl = rag_results['metrics'].get('rag_rouge_l', 0)
    rag_bs = rag_results['metrics'].get('rag_bertscore', 0)
    print(f"\n  [2] RAG Baseline (Ollama) → QLoRA Fine-tuned")
    print(f"  {'Metric':<18} {'RAG':>10} {'LoRA':>10} {'Delta':>10} {'Relative':>10}")
    print(f"  {'-'*60}")
    rl_d2 = lora_rl - rag_rl
    rl_r2 = (rl_d2 / rag_rl * 100) if rag_rl > 0 else 0
    bs_d2 = lora_bs - rag_bs
    bs_r2 = (bs_d2 / rag_bs * 100) if rag_bs > 0 else 0
    print(f"  {'ROUGE-L':<18} {rag_rl:>10.4f} {lora_rl:>10.4f} {rl_d2:>+10.4f} {rl_r2:>+9.1f}%")
    print(f"  {'BERTScore F1':<18} {rag_bs:>10.4f} {lora_bs:>10.4f} {bs_d2:>+10.4f} {bs_r2:>+9.1f}%")

print(f"\n  Note: LoRA 평가는 N={N_EVAL} 샘플, RAG 평가는 N={rag_results.get('n_eval', 'N/A') if rag_results else 'N/A'} 샘플 기준")
print("=" * 70)

---
## 12. Cross-Stage 연결: LoRA → GGUF 배포

### Fine-tuned 모델의 배포 과제

학습된 LoRA adapter는 그 자체로는 배포에 적합하지 않습니다:

1. **모델 병합 필요**: Base model + LoRA adapter → 단일 모델로 merge (`merge_and_unload`)
2. **메모리 제약**: FP16 Qwen2.5-7B ≈ 14GB → 일반 서버/데스크탑에서 운영 어려움
3. **추론 속도**: HuggingFace Transformers 추론은 최적화되지 않음 → 실시간 서비스에 부적합

### GGUF 양자화 (10) 해결책

| 과제 | GGUF 솔루션 |
|------|-----------|
| 모델 크기 | Q4_K_M: ~4GB (FP16 대비 ~70% 축소) |
| 추론 속도 | llama.cpp 최적화 → CPU에서도 10+ tokens/sec |
| 배포 편의 | 단일 .gguf 파일 + llama-cpp-python = 간단한 서빙 |
| 품질 보존 | K-quant 방식은 중요 레이어의 정밀도를 선택적으로 유지 |

**핵심 질문**: 양자화로 인한 품질 손실은 어느 정도인가? Q4 vs Q5 vs Q8의 품질-크기 트레이드오프를 10번 노트북에서 정량 비교합니다.

In [ ]:
# === Kaggle Dataset 자동 업로드 ===
if IS_KAGGLE:
    UPLOAD_DIR = '/kaggle/working/dataset_upload'
    os.makedirs(UPLOAD_DIR, exist_ok=True)

    # 결과 + LoRA adapter 심볼릭 링크
    for src in [results_path]:
        dst = os.path.join(UPLOAD_DIR, os.path.basename(src))
        if os.path.exists(dst):
            os.remove(dst)
        os.symlink(src, dst)

    # LoRA adapter 디렉토리 링크
    adapter_upload = os.path.join(UPLOAD_DIR, 'lora_adapter')
    if os.path.exists(adapter_upload):
        import shutil
        shutil.rmtree(adapter_upload)
    os.symlink(ADAPTER_DIR, adapter_upload)

    meta = {
        "title": "civilcomplaint-lora-adapter",
        "id": "kukass/civilcomplaint-lora-adapter",
        "licenses": [{"name": "CC0-1.0"}]
    }
    with open(os.path.join(UPLOAD_DIR, 'dataset-metadata.json'), 'w') as f:
        json.dump(meta, f, indent=2)

    !kaggle datasets create -p {UPLOAD_DIR} --dir-mode zip
    print("✅ Kaggle 데이터셋 업로드 완료: civilcomplaint-lora-adapter")
else:
    print("ℹ️ 로컬 환경 — Kaggle 업로드 건너뜀")

# --- GPU 메모리 정리 ---
del model, trainer
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print(f"GPU 메모리 해제 완료: {torch.cuda.memory_allocated()/1e9:.1f} GB 사용 중")